## Project: Question-Answering on Private Documents

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [2]:
def load_document(file):
    import os
    name, extension = os.path.splitext(file)
    if extension == '.pdf':
        from langchain_community.document_loaders import PyPDFLoader
        print(f"Loading {file}...")
        loader = PyPDFLoader(file)
    elif extension == '.txt':
        from langchain_community.document_loaders import TextLoader
        print(f"Loading {file}...")
        loader = TextLoader(file)
    elif extension == '.docx':
        from langchain_community.document_loaders import Docx2txtLoader
        print(f"Loading {file}...")
        loader = Docx2txtLoader(file)
    else:
        print(f"Unsupported file extension: {extension}")
        return None
    
    data = loader.load()
    return data

In [3]:
data = load_document('files/us_constitution.pdf')

Loading files/us_constitution.pdf...


In [5]:
print(data[1].page_content)

TheHouseofRepresentativesshallbecomposedofMemberschosen
everysecondYearbythePeopleoftheseveralStates,andthe
ElectorsineachStateshallhavetheQualificationsrequisitefor
ElectorsofthemostnumerousBranchoftheStateLegislature.
NoPersonshallbeaRepresentativewhoshallnothaveattainedtothe
AgeoftwentyfiveYears,andbeensevenYearsaCitizenoftheUnited
States,andwhoshallnot,whenelected,beanInhabitantofthatState
inwhichheshallbechosen.
RepresentativesanddirectTaxesshallbeapportionedamongthe
severalStateswhichmaybeincludedwithinthisUnion,accordingto
theirrespectiveNumbers,whichshallbedeterminedbyaddingtothe
wholeNumberoffreePersons,includingthoseboundtoServicefora
TermofYears,andexcludingIndiansnottaxed,threefifthsofallother
Persons.TheactualEnumerationshallbemadewithinthreeYears
afterthefirstMeetingoftheCongressoftheUnitedStates,andwithin
everysubsequentTermoftenYears,insuchMannerastheyshallby
Lawdirect.ThenumberofRepresentativesshallnotexceedonefor
everythirtyThousand,buteachStateshallhaveatLeastone
Rep

In [5]:
data = load_document('files/the_great_gatsby.docx')
print(data[0].page_content)

Loading files/the_great_gatsby.docx...
The Project Gutenberg eBook of The Great Gatsby, by F. Scott Fitzgerald



This eBook is for the use of anyone anywhere in the United States and

most other parts of the world at no cost and with almost no restrictions

whatsoever. You may copy it, give it away or re-use it under the terms

of the Project Gutenberg License included with this eBook or online at

www.gutenberg.org. If you are not located in the United States, you

will have to check the laws of the country where you are located before

using this eBook.



Title: The Great Gatsby



Author: F. Scott Fitzgerald



Release Date: January 17, 2021 [eBook #64317]

[Most recently updated: January 24 2021]



Language: English





Produced by: Alex Cabal for the Standard Ebooks project, based on a

             transcription produced for Project Gutenberg Australia.



*** START OF THE PROJECT GUTENBERG EBOOK THE GREAT GATSBY ***





			   The Great Gatsby

				  by

			 F. Scott Fitzger

In [1]:
# Loading from a website

def load_from_wikipedia(query, lang="en", num_pages=2):
    from langchain_community.document_loaders import WikipediaLoader
    loader = WikipediaLoader(query=query, lang=lang,load_max_docs=num_pages)
    data = loader.load()
    return data

In [2]:
data = load_from_wikipedia("GPT-4")
print(data[0].page_content)

Generative Pre-trained Transformer 4 (GPT-4) is a large language model developed by OpenAI and the fourth in its series of GPT foundation models.  
GPT-4 is preceded by GPT-3.5 and followed by its successor GPT-5. GPT-4V is a version of GPT-4 that can process images in addition to text. OpenAI has not revealed technical details and statistics about GPT-4, such as the precise size of the model.
An early version of GPT-4 was integrated by Microsoft into Bing Chat, launched in February 2023. GPT-4 was released in ChatGPT in March 2023, and removed in 2025. GPT-4 is still available in OpenAI's API.


== Background ==
 

OpenAI introduced the first GPT model (GPT-1) in 2018, publishing a paper called "Improving Language Understanding by Generative Pre-Training", which was based on the transformer architecture and trained on a large corpus of books. The next year, they introduced GPT-2, a larger model that could generate coherent text. In 2020, they introduced GPT-3, a model with over 100 ti

#### Chunking the loaded documents

In [7]:
# Loading US constitution
data = load_document('files/us_constitution.pdf')

from langchain_text_splitters import RecursiveCharacterTextSplitter
def chunk_data(data, chunk_size=256):
    print(f"Chunking the data into chunks of {chunk_size}...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size)
    chunks = text_splitter.split_documents(data)
    return chunks

Loading files/us_constitution.pdf...


In [8]:
chunks = chunk_data(data)
chunks[0].page_content

Chunking the data into chunks of 256...


'TheUnitedStatesConstitution\nWethePeopleoftheUnitedStates,inOrdertoformamoreperfect\nUnion,establishJustice,insuredomesticTranquility, provideforthe\ncommondefence,promotethegeneralWelfare,andsecurethe\nBlessingsofLibertytoourselvesandourPosterity, doordainand'

In [9]:
# Calculating the chunks cost
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model("text-embedding-ada-002")
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f"Total tokens: {total_tokens}")
    print(f"Cost: ${total_tokens / 1000 * 0.0004:.6f}")

In [10]:
print_embedding_cost(chunks)

Total tokens: 34048
Cost: $0.013619


#### Pinecone

In [3]:
from pinecone import Pinecone
from pinecone import ServerlessSpec
pc = Pinecone()

print(pc.list_indexes())

[{
    "name": "langchain",
    "metric": "cosine",
    "host": "langchain-z0ewswq.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}]


In [6]:
index_name = 'langchain'
if index_name not in pc.list_indexes():
    print(f"Creating index {index_name}...😊")
    pc.create_index(index_name, 
                    dimension=1536,
                    metric='cosine',
                    spec=ServerlessSpec(cloud='aws', region='us-east-1'))
    

Creating index langchain...😊


In [5]:
index_name="langchain"
pc.delete_index(index_name)

In [7]:
index = pc.Index(index_name)
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 14 Mar 2026 20:10:10 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '44',
                                    'x-pinecone-request-latency-ms': '43',
                                    'x-pinecone-response-duration-ms': '46'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


### Working with Vectors

In [8]:
# Create a dummy vector
import random
vectors = [[random.random() for _ in range(1536)] for _ in range(6)]
print(vectors)
print(len(vectors))


[[0.8185699884922129, 0.37310809834189496, 0.4594829376802545, 0.11673902983212792, 0.6270288223596104, 0.018621077414085474, 0.3744517152150061, 0.8599092439192875, 0.14323353373793324, 0.6165172390662529, 0.1397253118232178, 0.5567156155150954, 0.9973986410454309, 0.8773836877373551, 0.1444478674745373, 0.5003662287498614, 0.39294582883317564, 0.8038881508368368, 0.21757868056106722, 0.4228039445522651, 0.14662396696955482, 0.7754008044815515, 0.2996479498817425, 0.67658575208916, 0.5036672022049201, 0.6492110832216181, 0.5668622063715516, 0.5631588607149646, 0.182406738389833, 0.5502666686991616, 0.5730121502279755, 0.020696853687930394, 0.5481464298309247, 0.9284337983190857, 0.5574852232817824, 0.6169607028646732, 0.7799534889884996, 0.7472669434947602, 0.04226103138372361, 0.6181260490512015, 0.45662672416310734, 0.964618211712979, 0.42964651691654554, 0.9092511918754614, 0.27120829812200353, 0.12822893950297687, 0.3175989214668692, 0.56906103983108, 0.8101975779317655, 0.3185797

In [9]:
# To insert vector into a pinecone index we always need a unique id for each vector
ids = list('ABCDEF')
print(ids)

['A', 'B', 'C', 'D', 'E', 'F']


In [10]:
# Insert Vectors into Pinecone
index = pc.Index(index_name) # index_name is 'langchain' in this case.
index.upsert(vectors=zip(ids, vectors))

UpsertResponse(upserted_count=6, _response_info={'raw_headers': {'date': 'Sat, 14 Mar 2026 20:10:24 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '36870', 'x-pinecone-request-latency-ms': '307', 'x-envoy-upstream-service-time': '270', 'x-pinecone-response-duration-ms': '309', 'grpc-status': '0', 'server': 'envoy'}})

In [27]:
# Update Vectors
index.upsert(vectors=[('C', [0.5] * 1536)])

UpsertResponse(upserted_count=1, _response_info={'raw_headers': {'date': 'Sat, 14 Mar 2026 19:48:41 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '2', 'x-pinecone-request-logical-size': '6145', 'x-pinecone-request-latency-ms': '155', 'x-envoy-upstream-service-time': '156', 'x-pinecone-response-duration-ms': '158', 'grpc-status': '0', 'server': 'envoy'}})

In [45]:
# Fetch teh vectors using ids
index.fetch(ids = ['C'])

FetchResponse(namespace='', vectors={'C': Vector(id='C', values=[0.297014356, 0.910797298, 0.952552617, 0.0527521968, 0.282485485, 0.493733317, 0.454891264, 0.838137865, 0.110900238, 0.712962449, 0.240809247, 0.177449048, 0.314281046, 0.375928164, 0.509539, 0.0688088164, 0.474001437, 0.518259525, 0.184493378, 0.281516016, 0.278062403, 0.543731034, 0.928039849, 0.249688804, 0.105138034, 0.343864977, 0.164043307, 0.158028677, 0.978265285, 0.645818293, 0.900124788, 0.272182077, 0.366401523, 0.947392464, 0.0680151433, 0.0467136465, 0.525850177, 0.438611478, 0.732218266, 0.661863923, 0.0419263802, 0.26662761, 0.981445193, 0.197938025, 0.505569398, 0.867423594, 0.0505396463, 0.236125842, 0.63440603, 0.050045114, 0.70058614, 0.960344374, 0.770127237, 0.192277104, 0.504472, 0.87561661, 0.146374151, 0.469505548, 0.210130602, 0.93014431, 0.697584271, 0.730053246, 0.905913532, 0.475434691, 0.72005707, 0.0614006221, 0.827230334, 0.866282463, 0.391187459, 0.846477449, 0.774525344, 0.982027113, 0.73

In [29]:
# Delete Vectors using ids
index.delete(ids = ['A', 'C'])

{'_response_info': {'raw_headers': {'date': 'Sat, 14 Mar 2026 19:52:26 GMT',
   'content-type': 'application/json',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-pinecone-request-lsn': '3',
   'x-pinecone-request-logical-size': '0',
   'x-pinecone-request-latency-ms': '154',
   'x-envoy-upstream-service-time': '155',
   'x-pinecone-response-duration-ms': '156',
   'grpc-status': '0',
   'server': 'envoy'}}}

In [36]:
pc.delete_index(index_name)

In [46]:
# Querying the vectors using Pinecone
X = [random.random() for _ in range(1536)]

index.query(
    vector=X,
    top_k=2,
    include_values=False
)


QueryResponse(matches=[{'id': 'F', 'score': 0.765693307, 'values': []}, {'id': 'B', 'score': 0.765419245, 'values': []}], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Sat, 14 Mar 2026 19:57:48 GMT', 'content-type': 'application/json', 'content-length': '151', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '39', 'x-envoy-upstream-service-time': '38', 'x-pinecone-response-duration-ms': '40', 'grpc-status': '0', 'server': 'envoy'}})

### Namespace

Pinecone allows you to group the vectors in an index using a namespace.  
You can insert vectors into a particular namespace.

In [33]:
# Create a dummy vectors list to insert into a namespace
first_namespace_vector = [[random.random() for _ in range(1536)] for _ in range(3)]
ids = list('XYZ')
index.upsert(vectors=zip(ids, first_namespace_vector), namespace="first-namespace")

UpsertResponse(upserted_count=3, _response_info={'raw_headers': {'date': 'Sat, 14 Mar 2026 20:44:27 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '18435', 'x-pinecone-request-latency-ms': '262', 'x-envoy-upstream-service-time': '241', 'x-pinecone-response-duration-ms': '265', 'grpc-status': '0', 'server': 'envoy'}})

In [18]:
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '256',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 14 Mar 2026 20:16:35 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '38',
                                    'x-pinecone-request-latency-ms': '37',
                                    'x-pinecone-response-duration-ms': '39'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 6},
                'first-namespace': {'vector_count': 3},
                'second-namespace': {'vector_count': 2}},
 'storageFullness': 0.0,
 'total_vector_count': 11,
 'vector_type': 'dense'}


In [19]:
second_namespace_vectors = [[random.random() for _ in range(1536)] for _ in range(2)]
ids = list('qp')
index.upsert(vectors=zip(ids, second_namespace_vectors), namespace="second-namespace")

UpsertResponse(upserted_count=2, _response_info={'raw_headers': {'date': 'Sat, 14 Mar 2026 20:16:45 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '2', 'x-pinecone-request-logical-size': '12290', 'x-pinecone-request-latency-ms': '162', 'x-envoy-upstream-service-time': '160', 'x-pinecone-response-duration-ms': '165', 'grpc-status': '0', 'server': 'envoy'}})

In [24]:
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '256',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 14 Mar 2026 20:39:07 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '76',
                                    'x-pinecone-request-latency-ms': '75',
                                    'x-pinecone-response-duration-ms': '77'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 6},
                'first-namespace': {'vector_count': 3},
                'second-namespace': {'vector_count': 2}},
 'storageFullness': 0.0,
 'total_vector_count': 11,
 'vector_type': 'dense'}


In [34]:
# Now if we try to fetch without namespace we wont get the vector 
index.fetch(ids=['X'])
# Returns empty vector since namespace is not specified

FetchResponse(namespace='', vectors={}, usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Sat, 14 Mar 2026 20:44:30 GMT', 'content-type': 'application/json', 'content-length': '53', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '43', 'x-envoy-upstream-service-time': '44', 'x-pinecone-response-duration-ms': '46', 'grpc-status': '0', 'server': 'envoy'}})

In [35]:
index.fetch(ids=['X'], namespace="first-namespace")
# Return the vector since the namespace is specified

FetchResponse(namespace='first-namespace', vectors={'X': Vector(id='X', values=[0.694523156, 0.743951142, 0.948732138, 0.481702715, 0.784382403, 0.452284127, 0.748503208, 0.599654853, 0.639220238, 0.329078227, 0.880702078, 0.846613884, 0.885774493, 0.90573293, 0.630902529, 0.371816844, 0.283643454, 0.367147952, 0.282084644, 0.628248096, 0.687565, 0.300897807, 0.512023211, 0.182919055, 0.536466062, 0.751613319, 0.608364582, 0.367123693, 0.0343759805, 0.844952583, 0.288452178, 0.96592474, 0.71263957, 0.441445291, 0.204395264, 0.614044905, 0.981666565, 0.207591698, 0.0897039101, 0.144530758, 0.931817651, 0.533084929, 0.531002045, 0.76954484, 0.0385610797, 0.579894066, 0.313770235, 0.184414044, 0.73138, 0.204308, 0.219413877, 0.817530394, 0.546598434, 0.956417263, 0.892333269, 0.957624137, 0.779782295, 0.383008689, 0.227070808, 0.100199692, 0.532412708, 0.929006696, 0.0797236115, 0.843244731, 0.117164865, 0.269881278, 0.137214571, 0.316829652, 0.813088238, 0.53969866, 0.565203428, 0.387859

In [36]:
index.delete(ids=['X'], namespace="first-namespace")
# Delete works with the same logic.

{'_response_info': {'raw_headers': {'date': 'Sat, 14 Mar 2026 20:44:41 GMT',
   'content-type': 'application/json',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-pinecone-request-lsn': '2',
   'x-pinecone-request-logical-size': '0',
   'x-pinecone-request-latency-ms': '156',
   'x-envoy-upstream-service-time': '156',
   'x-pinecone-response-duration-ms': '157',
   'grpc-status': '0',
   'server': 'envoy'}}}

In [37]:
# Delete everything in a namespace 
index.delete(delete_all=True, namespace='first-namespace')


{'_response_info': {'raw_headers': {'date': 'Sat, 14 Mar 2026 20:45:00 GMT',
   'content-type': 'application/json',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-pinecone-request-latency-ms': '96',
   'x-envoy-upstream-service-time': '96',
   'x-pinecone-response-duration-ms': '97',
   'grpc-status': '0',
   'server': 'envoy'}}}